# model_05 — deneme 2 · GERÇEK VERİ

Kullanıcı kararı, 17 Eylül 2026: *"comp çalışan bir modelin benim
nezdimde ANLAMLI BİR VERİDE çalıştığını test etmek. çok daha mantıklı
ve gerçek hayatı daha çok yansıtan bir veri tabanı olacak."*

Önceden kayıt `belge/onkayit/model_05.md`.

### DÜĞME — VERİ

`model_03`ün reçetesi (`wd 0.5` + sabit LR) aynen devralınıyor. Mimari,
optimizasyon, bütçe, bölme oranları — hepsi aynı. Değişen tek şey
**veri ve onun dili**.

```
                    model_03 (veri_04)        model_05 (veri_05)
varlik              1060                      1087
tip                 4   KISI OKUL SEHIR DERS  7   + UNIVERSITE FAKULTE BOLUM BOLGE
iliski              17                        25
olgu                10.960                    6.254
egitim 2-hop        68.436                    24.962
phi                 6,24                      3,99
```

### `veri_04` NEYDİ, `veri_05` NEYİ DÜZELTİYOR

`veri_04` yüksek phi'yi şemayı zorlayarak alıyordu — ölçüldü:

```
"Matematik dersinin kurucusu kim"        400 / 10.960 olgu
"Ankara sehrinin dersi ne"               kendi icinde CELISKILI
200/200 OKUL adi kendi sehrini SIZDIRIYOR (Ankara_Lisesi -> Ankara)
"Ali Yildiz'in kardesinin kardesi"       AYNI havuzdan RASTGELE
"Fatma Dogan'in annesi Huseyin Dogan"    cinsiyet TUTMUYOR
```

`veri_05`te bunların hiçbiri yok. Şema gerçek bir üniversite
hiyerarşisi, aile gerçek bir soy ağacı (cinsiyet ve kuşak tutarlı,
ensest yapısal olarak imkânsız), şehir–bölge eşlemesi gerçek coğrafya.
11 denetim bunu her koşuda sınıyor.

Fakülte/bölüm adları **nötr**: öneki üstündeki varlığın adıyla ilgisiz
(`Cerrahpaşa Mühendislik Fakültesi` → `Boğaziçi Üniversitesi`), yani
cevabın yarısı soruda durmuyor. Aile soyadı ise **bilerek** paylaşılıyor
— gerçek hayatta da öyle — ve `sizinti()` onu zincir zincir ölçüyor.

### DİL — ikinci düğme, ve AYRILAMAZ

Kullanıcı, 17 Eylül: *"ben düzgün bir Türkçe ile eğitim istiyorum."*
Kodlama baştan aşağı yeniden yazıldı (`ek_kip` "tr" → "tr2"):

```
ONCE                                  SIMDI
@kardesi <SI>   iliski + soyut ek     kardesi           kelimenin KENDISI
<NIN>           tek jeton, 8 allomorf 'in 'in 'un 'un   GERCEK ekler
                                      nin nin nun nun
<DIR>           tek jeton             dir dir dur dur / tir tir tur tur
`?` ile biter                         kim / neresi / hangisi  soru sozcugu
<YOK> dolgusu   jetonlarin %11,5'i    YOK -- sinirI kesme isareti tasir
[S1] [S2] [KIMLIK] + 2 bos yuva       SILINDI (besi de HIC gecmiyordu)
devrik bicim    "kardesi Ibrahim'in"  "Kim Ibrahim Yilmaz'in kardesi?"
cevap = varlik                        cevap = TAM CUMLE
```

Satır şu hâle geldi:

```
Ibrahim Yilmaz ' in kardesi kim ? Ibrahim Yilmaz ' in kardesi Ozlem Yilmaz ' dir .
  "Ibrahim Yilmaz'in kardesi kim?  Ibrahim Yilmaz'in kardesi Ozlem Yilmaz'dir."

t_len 17 -> 24      vocab 301 -> 289 (289 jetonun 288'i havuzda GECIYOR)
```

Her olgu üç yüzey biçiminde: KANONİK (soru), DEVRİK (soru sözcüğü başta),
BİLDİRİM (soru yok, olgu doğrudan). Üçünde de cevap cümlesi aynı, yani
olgu üç bağlamda da aynı geçişten öğreniliyor.

> **ATFETME YAPILAMAZ.** Veri VE dil birlikte değişiyor. İkisi
> ayrılamaz: yeni şema eski kodlamayla, eski şema yeni kodlamayla
> yazılamaz. CLAUDE.md: *"iki düğme birden → ATFETME yapılamaz diye
> YAZILIR, kol koşulur."*

### KIYAS SÜTUNU YOK — ve bu eksiklik değil

`model_03` ve öncesi hep aynı sınavda koştu (ölçme izi
`d6751004648c`). `veri_05` tam da o sınavı değiştiriyor; yeni iz
`f4ce53fd1555`. Eski sayıları aynı tabloya koymak başka bir sınavın
sonucunu oraya koymak olurdu.

```
model_03 PENCERE 12-20k    one 0,9857  seen 0,9960  comp 0,8350  ent 0,0423
                           ^ BASKA SINAV. Yonelme icin; fark ALINMAZ.
```

Kullanıcı, 17 Eylül: *"ben kıyas derdinde değilim."* CLAUDE.md:
*"bir kolun değeri comp ve ent ne oldu."* **Kapılar gevşemedi** —
gevşeyen yalnız kıyas.

### KARAR KURALI — son pencere

**BİRİNCİL ÖLÇÜ `comp`.** Soru: *reçete gerçek ve tutarlı bir Türkçe
veride de bileşimi açıyor mu?*

```
comp               HUKUM
>= 0,50            OLGUNLUK KAPISI GECTI -- recete gercek veriye TASINDI
0,30 - 0,50        BELIRGIN ama kapi altinda
0,15 - 0,30        kismi
<= 0,15            recete BU VERIDE CALISMIYOR

ON KOSUL (KORUMA)  one >= 0,98  VE  seen >= 0,95
                   gecmezse `comp` YORUMLANMAZ -- taban beceri eridiyse
                   bu KAZANC degil TAKASTIR

BIRIM TESTI        ent_yok_kisayol == 0,000   <- KIRMIZI CIZGI
```

`ent`, `ood` ve `ent_kisayol` **raporlanır, hüküm vermez**.

> `ood` bu kolda **70 örnek** (model_03'te 210). Tek örnek 0,014
> oynatıyor — gürültülü, hükümde kullanılmaz.

> **SAYISAL TAHMİN YAZILMIYOR** (kullanıcı kararı, 15 Eylül).

### BİLİNEN EKSİK — önkayda yazıldı

`analiz_05`in 39 denetiminden biri düşüyor: BÖLÜM, FAKÜLTE ve BÖLGE
tiplerinin hiç `ent` zinciri yok, çünkü o üç tipin hiçbir ilişkisi
kısayolu tip olarak mümkün kılacak biçimde örtüşmüyor. Yani tutulmuş
sınav 7 tipin 4'ünü sınıyor. Kapatılmadan koşuluyor ve `ent` okunurken
bu akılda tutulur.

### Beklenen kilit çıktısı

```
model_05/test_05.py    155 gecti, 0 BOZUK
graf_05                tip ihlali 0
havuz_05               HEPSI GECTI
```

---

**Sıra:** 0 → 1 → 2 → 3 → 4 ile başlat. Koşu sürerken **5 ve 6**.
Bitince **7** (`pencere_05`) ve **8** (`tani_05`).


In [ ]:
# 0 MODEL ADI VE YOLLAR  |  CPU  |  tekrar: GUVENLI
MODEL = "model_05"            # <-- DEGISTIRILECEK TEK SATIR

assert MODEL, ("MODEL bos. Bu SABLON -- kopyala, adini modelin adi yap "
               "(model_a.ipynb) ve bu satiri doldur.")
import time
DEPO  = "https://github.com/sekerahmet/sekerai.git"
KOD   = "/content/kod"
EV    = f"/content/drive/MyDrive/{MODEL}"     # TEPE KLASOR = MODELIN ADI
# LOG ADI BURADA URETILMEZ -- 4. hucre her BASLATMADA kendi damgasini
# basar. Sebep olculdu (15 Eylul): log adi burada uretilince, bu hucreyi
# yeniden calistirmadan ikinci bir kosu baslatmak ONCEKI kosunun logunu
# "w" ile ACIP SIFIRLIYOR. Fiilen oldu: 20.000'lik kosunun logu, 40.000'lik
# kosu baslayinca silindi. Damga BASLATAN hucrede uretilirse imkansiz.
print(MODEL, "->", EV)

In [ ]:
# 1 GPU VAR MI, BOS MU  |  GPU'yu SORAR, kullanmaz  |  tekrar: GUVENLI
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_p = torch.cuda.get_device_properties(0)
_bos = torch.cuda.mem_get_info()[0] / 1e9
print(f"{_p.name}   toplam {_p.total_memory/1e9:.1f} GB   bos {_bos:.1f} GB")
assert _bos > 3.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"

In [ ]:
# 2 DRIVE  |  CPU  |  tekrar: GUVENLI (yalniz <model>/log/ acar)
from google.colab import drive
import os
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), \
    "drive.mount CALISMADI -- /content/drive gercek bir baglanti degil."
os.makedirs(f"{EV}/log", exist_ok=True)
with open(f"{EV}/.yazma_denemesi", "w") as f:
    f.write("ok")
os.remove(f"{EV}/.yazma_denemesi")
print("hazir ve YAZILABILIR:", EV)
print("  var olan tohum klasorleri:",
      sorted(d for d in os.listdir(EV) if d.startswith("t")) or "(yok)")

In [ ]:
# 3 KODU GITHUB'DAN CEK + KILIT TESTI  |  CPU  |  tekrar: KOSU YOKKEN (ilk isi rm -rf /content/kod)
import subprocess, os, glob, importlib, sys
subprocess.run(["rm", "-rf", KOD], check=True)
subprocess.run(["git", "clone", "--depth", "1", DEPO, KOD], check=True)
COMMIT = subprocess.run(["git", "-C", KOD, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT, "|",
      subprocess.run(["git", "-C", KOD, "log", "-1", "--format=%s"],
                     capture_output=True, text=True).stdout.strip())

# Her modelin AILE klasoru var: deneme2/model_a/{model_a.py, pencere_a.py,
# model_a.ipynb}. Klasor adini isimden TURETMIYORUZ, dosyayi ARIYORUZ --
# model_a1 gibi varyasyonlar da ayni aile klasorunde durur.
# !! model_05 ARANMAZ: kendi klasoru BELLI. Kullanici karari,
# 16 Eylul -- "model_05 diger hicbir model ile ayni seyi kullanmamali".
# Paylasilan sablon `deneme2/*/<MODEL>.py` glob'u yapiyordu; model_05
# artik o aramaya girmiyor.
_aday = glob.glob(f"{KOD}/deneme2/model_05/{MODEL}.py")
assert len(_aday) == 1, f"{MODEL}.py tam bir kez bulunmali, bulunan: {_aday}"
AILE = os.path.dirname(_aday[0])
_pen = glob.glob(f"{AILE}/pencere_*.py")
assert len(_pen) == 1, f"ailede tam bir pencere_*.py olmali: {_pen}"
PENCERE = _pen[0]
print("aile:", AILE, "| olcum:", os.path.basename(PENCERE))

# --- IMPORT ONBELLEGINI TEMIZLE -- BU HUCRENIN EN SESSIZ TUZAGI ----------
# `rm -rf` + yeniden klon KODU tazeler ama `sys.modules` ESKI modul
# nesnesini tutar. Ayni cekirdekte MODEL degistirip bu hucreyi yeniden
# kosarsan, yeni kodu klonlamis ama ESKI modulu kullaniyor olursun.
# OLCULDU (15 Eylul): model_a4'ten model_a5'e gecerken
#   "AssertionError: Ayar'da boyle alan yok: {'ort_bas'}"
# cikti -- cunku model_a hala onceki klondan gelen, `ort_bas`i olmayan
# nesneydi. Daha sinsi hali: alan adlari tutarsa hata VERMEZ ve kosu
# ESKI KODLA baslar; kunyedeki commit ise YENIYI gosterir.
_atilan = [n for n, m in list(sys.modules.items())
           if getattr(m, "__file__", None) and str(m.__file__).startswith(KOD)]
for n in _atilan:
    del sys.modules[n]
importlib.invalidate_caches()
if _atilan:
    print("import onbellegi temizlendi:", sorted(_atilan))

# --- KILIT: test_05.py ---------------------------------------------------
# model_05 KENDI motoruna (taban_05.py) ve KENDI verisine (veri_05.py)
# sahip -- ikisi de model_a / veri_okul KOPYASI. `test_05.py` dort sey
# tutuyor: (0) BAGIMSIZLIK, model_05/*.py disariya import ETMIYOR;
# (1) MIMARI; (2) VERI, veri_05 KENDI iddialarini tutuyor
#     (sema, soy agaci, zincir siniflari, notrluk); (3) MOTOR,
# egitim havuzu ve olcme izi model_a ile AYNI.
# Duserse egitim BASLAMAMALI -- sayilar baska bir tabloda okunur.
#
# CIKTI YAKALANIR ve BASILIR: Colab alt surec stdout'unu hucreye
# aktarmiyor; "cikti yok" ile "test kosmadi" ayrimi sansa birakilmaz.
# AYRINTI=1 -> gecen kontroller de basilir (dugme tablosu gorunur olsun).
_t = [x for x in glob.glob(f"{AILE}/test_*.py")]
if _t:
    print()
    print("=" * 72)
    _r = subprocess.run([sys.executable, _t[0]], cwd=AILE,
                        capture_output=True, text=True,
                        env={**os.environ, "AYRINTI": "1",
                             "KOSU_KOK": EV.rsplit("/", 1)[0]})
    print(_r.stdout.rstrip() or "(cikti YOK -- test kosmamis olabilir!)")
    if _r.stderr.strip():
        print("stderr:", _r.stderr.rstrip()[-2000:])
    print("=" * 72)
    assert _r.returncode == 0, (
        f"{os.path.basename(_t[0])} DUSTU (cikis {_r.returncode}). Taban "
        f"degismis olabilir ve KAYITLI sonuclar ona dayaniyor. EGITIM BASLATMA.")
    print()

sys.path.insert(0, AILE)
# !! UST KLASOR EKLENMIYOR: veri modulu de (veri_05.py) AILE icinde.
# model_05 `deneme2/` kokundeki hicbir seyi gormez.
M_05 = importlib.import_module(MODEL)
M = M_05
# `M_05.M` MOTOR (taban_05). Paylasilan sablonda `M` TABANI
# gosteriyordu (model_a); burada `M` kolun KENDISI, motor ise
# `M_05.M`. Miras denetimi taban sinifi ORADAN alir.
MOTOR = M_05.M
assert M.AYAR.ad == MODEL, f"AYAR.ad {M.AYAR.ad!r} != dosya adi {MODEL!r}"
# Yuklenen modul GERCEKTEN yeni klondan mi geldi?
assert M.__file__.startswith(KOD), f"{MODEL} {KOD} disindan geldi: {M.__file__}"
# BU KOLUN SARTLARI. Hicbiri DEVRALINMIYOR: `ayar_05.py` her alani
# `Ayar()` varsayilaninin uzerine tek tek, gerekcesiyle yaziyor.
# --- MIMARI: bu kolun TANIMI --------------------------------------
assert M.AYAR.dongu == 1,              "DONGU YOK"
assert M.AYAR.l == 8,                  "8 AYRI katman"
assert M.AYAR.dar_alfa == 0.0,         "Phi DARBOGAZI YOK"
assert M.AYAR.dar_kapi is False,       "ogrenilen GECIT YOK"
assert M.AYAR.dff == 704,              "SwiGLU d_ff = 8/3*d"
assert M.AYAR.d // M.AYAR.nh == 64,    "head_dim 64"
# KUSUR (16 Eylul, defterin ILK kosusu): burada `M.Model` yaziyordu
# ve `M` = model_05 modulu; onda `Model` YOK -> AttributeError.
# Sablondan devralinmisti, defter hic kosulmadigi icin gorulmedi.
assert not issubclass(M_05.ModelSade, MOTOR.Model), \
    "ModelSade taban_05.Model den MIRAS ALMAMALI"
# ======================================================================
# BURADAN ASAGISI MIMARI DEGIL. Uc AYRI kategori, karistirilmasin:
#
#   A) DILIN KENDISI (korpus)  -- hiperparametre DEGIL. "ek_kip=tr"
#      demek "bu dil ekli bir dil" demek; "bicim=3" demek "ayni olgu
#      uc yuzey biciminde geciyor" demek. Modelin secimi degil,
#      METNIN ozelligi.
#   B) SINAV BOLMELERI          -- olcumun tanimi, modele ait DEGIL.
#   C) PROJEYE OZGU VERI EKI    -- BEST PRACTICE DEGIL. Tek kalem:
#      identity bridge (ident_frac / ident_kip). model_b13'te
#      arXiv 2509.24653'ten alindi. Burada DURUYOR cunku egitim havuzu
#      model_b15 ile AYNI KALMASI BEKLENEN alanlar. model_05
#      IKISINI bilerek ayiriyor (veri_ad, ek_kip) -- bu kolun
#      TANIMI. test_05 bunlari bildirilmis ayrisma diye isler.
#      (onkayit model_05.md 2).
#
# Optimizasyon (wd/cosine/isinma/lr/betas) YUKARIDA degil ASAGIDA ve
# hepsi STANDART TARIF -- proje kisiti DEGIL.
# ======================================================================
# --- A) DILIN KENDISI -------------------------------------------------
# --- BU KOLUN TEK DUGMESI ------------------------------------------
assert M.AYAR.wd == 0.5,               "model_a3 RECETESI -- kolun TANIMI"
assert M.AYAR.sabit_lr is True,        "LR SABIT -- recetenin ikinci yarisi"
assert not hasattr(M.AYAR, "dusun_gecis"), "model_05te DONGU YOK"
assert M.AYAR.veri_ad == "veri_05",    "model_05 KENDI veri modulu"
import veri_05 as _V05
assert _V05.graf_izi(_V05.kur(0)) == _V05.IZ, "graf KAYMIS"
_G05 = _V05.kur(0)
_nv, _no = sum(_G05["n"].values()), len(_G05["olgu"])
print(f"veri_05  graf izi {_V05.IZ}   {_nv} varlik  {_no} olgu  "
      f"|R| {len(_V05.ILISKI)}  {len(_V05.TIPLER)} tip"
      f"   (universite hiyerarsisi + gercek soy agaci)")
assert M.AYAR.jeton_ad == "tam",       "varlik = JETON DIZISI (1-3 kelime)"
assert M.AYAR.ek_kip == "tr2",         "dil GERCEK TURKCE ekli"
# tr2: iliski KELIMENIN KENDISI (kucuk harf: fakultesi), ek
# jetonlari GERCEK allomorflar ('in/'in/'un/'un/'nin...),
# soru sozcugu VAR (kim/neresi/hangisi), <YOK> dolgusu YOK.
assert M.AYAR.bicim == 3,              "ayni olgu UC yuzey biciminde"
assert M.AYAR.t_len == 24,             f"t_len = 3*yuva + 15: {M.AYAR.t_len}"
assert M.MOTOR.SPECIAL == 3,           "olu ozel jeton YOK"
assert M.AYAR.belge_pay == 0.0,        "BELGE satiri YOK (ek_kip ile kurulmadi)"

# --- B) SINAV BOLMELERI -- olcumun tanimi -----------------------------
assert M.AYAR.ood_pay == 0.05,         f"ood bolmesi: {M.AYAR.ood_pay}"
assert M.AYAR.kopru_kayip == 0.0,      "YARDIMCI KAYIP YOK (bu MIMARI karari)"

# --- C) PROJEYE OZGU VERI EKI -- BEST PRACTICE DEGIL ------------------
# Kullanici, 16 Eylul: "biz her seyi sifirdan yaptik, ben kisit
# vermedim, best practice dedim." Dogru -- ve bu IKI SATIR o tarifin
# parcasi DEGIL. Egitim havuzu b15 ile ayni kalsin diye duruyor.
assert M.AYAR.ident_frac == 0.2,       "identity bridge -- PROJE EKI, b15 ile AYNI"
assert M.AYAR.ident_kip == "q1",       "identity bridge -- PROJE EKI, b15 ile AYNI"

# --- OPTIMIZASYON: STANDART TARIFIN KENDISI ---------------------------
# Bunlar proje kisiti DEGIL. Dordu de ayni sayilari kullaniyor:
#   nanoGPT   GPT-3   Llama   Pythia
assert M.AYAR.lr == 1e-3,              "Pythia-70m ile ayni mertebe"
assert M.AYAR.isinma == 2000,          "nanoGPT warmup_iters=2000, Llama 2000"
assert M.AYAR.betas == (0.9, 0.95),    "nanoGPT/GPT-3/Llama/Pythia -- 0.999 DEGIL"
assert M.AYAR.tam_kayip is True,       "butun pozisyonlarda next-token: standart LM"
# PROJEYE OZGU OPTIMIZASYON NUMARALARI -- HEPSI KAPALI
assert M.AYAR.ort_bas == 0,            "LOOKAHEAD KAPALI -- hicbir tarifte YOK"
assert not (M.AYAR.dar_sert or M.AYAR.dar_sdpa), "darbogaz zaten YOK"
print("mimari: ModelSade  8 katman  RoPE  SwiGLU  bagli gomme  bias YOK")
SOR = os.path.join(AILE, "sor_05.py")  # 9. hucre kullanir
for _g in ("AYAR", "egit", "fark_bas"):
    assert hasattr(M, _g), f"{MODEL}'de {_g} YOK -- kos_05.py duser"
print("ayar:", M.AYAR)

In [ ]:
# 4 BASLAT  |  GPU (alt surec)  |  tekrar: HAYIR -- yeni kosu baslatir
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
# GPU kullanacak her hucre, ONCE kendi icinde GPU'yu sorar. Ayri bir
# "GPU var mi" hucresi HUCRE SIRASINA bagli bir kuraldir; icerideki
# kontrol degildir. Olculdu (15 Eylul): Colab cekirdegi kosu sirasinda
# oldu, GPU ve Drive durumu gitti; ayri kontrol hucresi vardi ama
# calistirilmamisti.
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------

TOHUMLAR = [0]             # ONCE TEK TOHUM.
# !! BU KOLUN OLCUTU `comp` (ent DEGIL) -- onkayit model_05.md.
# Sonuc OLUMLU cikarsa (comp yukseldiyse) tek tohum YETER.
# OLUMSUZ cikarsa bu satiri [1, 2] yapip tekrar kos -- t0 klasoru
# DOKUNULMAZ, yeni kosular t1/ ve t2/'ye yazar. Kod degismez.
# Neden onemli: grokking tohuma bagli. Tek tohumda comp yukselmezse
# "konfigurasyon yanlis" ile "bu baslangic sanssiz" AYRILAMAZ.
# VERI tohumu AYRI (ayar.veri_tohum=0) -- butun tohumlar AYNI veriyi gorur.

# !! ADIM buyutup SURDUR=True yapmak UZATMADIR, sifirdan kosu DEGIL.
#    Ama cosine ufku `ayar.adim`dan turedigi icin LR GERI FIRLAR:
#    model_b14'te 20.000 ufkunda lr/10 iken 60.000 ufkunda 7,6 KAT.
#    Uzatilmis kosu, temiz bir uzun kosu DEGILDIR.
ADIM   = None    # None = ayar_05.py'deki ILK SINIR (20.000).
#                  Uzatmak KULLANICI karari (CLAUDE.md kural 1).
SURDUR = False   # TAZE kosu -- surdurme DEGIL.
USTUNE = False   # t0 BOS (yeni kol). True = dolu klasoru
#                  t<N>_eski_<zaman>/'a TASI (silmez)

if "p" in globals() and p.poll() is None:
    raise SystemExit(f"ZATEN KOSUYOR (PID {p.pid}). Once 'Durdurmak' hucresi.")

# KENDI kosucusu: model_05/kos_05.py. `--model` YOK -- bu betik
# yalnizca model_05'i baslatir (paylasilan kos.py aile klasoru ARIYORDU).
_arg = [sys.executable, "-u", f"{KOD}/deneme2/model_05/kos_05.py",
        "--ev", EV, "--commit", COMMIT,
        "--tohum", *[str(t) for t in TOHUMLAR]]
if ADIM:
    _arg += ["--adim", str(ADIM)]
if SURDUR:
    _arg += ["--surdur"]
if USTUNE:
    _arg += ["--ustune"]
# LOG ADI HER BASLATMADA YENI: bu hucre iki kez calisirsa iki AYRI log
# olur, oncekinin uzerine YAZILMAZ.
LOG = f"{EV}/log/kos_{time.strftime('%Y%m%d_%H%M%S')}.txt"
p = subprocess.Popen(_arg, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("PID", p.pid, " tohum", TOHUMLAR, " -> log:", LOG)
print("Dolu bir tohum klasoru varsa kosu REDDEDILIR -- hicbir sey ezilmez.")
print("Ayni tohumu bilerek tekrar kosmak icin --ustune; o da SILMEZ,")
print("eskisini t<N>_eski_<zaman>/ diye yan klasore TASIR.")

In [ ]:
# 5 ILERLEME (ham log)  |  CPU  |  tekrar: GUVENLI (log geriden gelebilir)
import glob, subprocess
# LOG degiskenine DEGIL, klasordeki EN YENI log'a bak.
_l = sorted(glob.glob(f"{EV}/log/kos_*.txt"))
assert _l, f"log yok: {EV}/log/"

# `p` CEKIRDEK YENIDEN BASLAYINCA KAYBOLUR -- ve tam o an bu hucreye
# ihtiyac duyulur. Olculdu (15 Eylul): Colab cekirdegi oldu, bu hucre
# `NameError: name 'p' is not defined` verdi, koşunun yasayip yasamadigi
# ogrenilemedi. Artik `p` yoksa SUREC TABLOSUNA bakiyor.
if "p" in globals():
    _d = p.poll()
    print("KOSUYOR" if _d is None else f"BITTI/OLDU (cikis kodu {_d})",
          "| PID", p.pid)
else:
    _ps = subprocess.run(
        ["bash", "-lc", "ps -eo pid,etime,cmd | grep kos_05.py | grep -v grep"],
        capture_output=True, text=True).stdout.strip()
    print("!! `p` YOK -- cekirdek yeniden baslamis.")
    if _ps:
        print("   ama SUREC YASIYOR:\n   " + _ps)
    else:
        print("   ve kos_05.py sureci de YOK -> kosu OLDU.")
        print("   Drive'i yeniden bagla (2. hucre), 3'u kos, sonra 4. hucrede")
        print("   SURDUR = True ile KALDIGI YERDEN devam ettir.")
print("log:", _l[-1].split("/")[-1], f"({len(_l)} log dosyasi var)")
print("-" * 78)
print(open(_l[-1]).read()[-4000:])

In [ ]:
# 6 RAPOR (canli durum)  |  CPU  |  tekrar: GUVENLI
import json, glob, os, statistics

# !! BU KOLUN KIYAS SUTUNU YOK -- ve bu bir eksiklik DEGIL.
#
# Hucre model_03'ten kopyalandiginda dort sutun basiyordu: model_00,
# model_b15, model_b8 ve bu kol. Hepsi AYNI SINAVDA kosmustu
# (olcme izi d6751004648c) ve sayilar satir satir okunabiliyordu.
#
# `veri_05` tam da o sinavi degistiriyor: yeni sema (27 iliski, 7 tip),
# gercek soy agaci, gercek cografya, ve GERCEK TURKCE kodlama.
# Eski sutunlari burada tutmak, BASKA BIR SINAVIN sayilarini ayni
# tabloya koymak olurdu -- model_b8 sutunu model_03'te tam bu yuzden
# "b8 !BSK" diye isaretlenmisti. Simdi HEPSI o durumda, o yuzden
# hepsi CIKTI.
#
# Kullanici, 17 Eylul: *"ben kiyas derdinde degilim. comp calisan bir
# modelin benim nezdimde ANLAMLI BIR VERIDE calistigini test etmek."*
# CLAUDE.md: "kiyas artik arka planda ... bir kolun degeri comp ve ent
# ne oldu." Kapilar GEVSEMEDI -- gevseyen yalniz kiyas.
#
# Yonelme icin eski kollarin sayilari ASAGIDA duruyor ama TABLOYA
# GIRMIYOR; basligi da BASKA SINAV diyor.
ESKI = {                       # BASKA SINAV (iz d6751004648c, veri_04)
    "model_00": dict(one=1.0000, seen=1.0000, comp=0.1303, ent=0.0417),
    "model_03": dict(one=0.9857, seen=0.9960, comp=0.8350, ent=0.0423),
}
IZ_05 = "f4ce53fd1555"         # model_05'in KENDI sinavi
#   d6751004648c -> 010062a47e31 -> f4ce53fd1555
#   birincisi veri_04, ikincisi veri_05 + eski kodlama,
#   ucuncusu veri_05 + "tr2" (gercek ekler, soru sozcugu, dolgusuz ad).
M03_MS = 77.0                  # model_03 OLCULDU (t_len 17) -- HIZ kiyaslanir,
#                                cunku mimari ve dizi uzunlugu AYNI kaldi.

_TUM = (("adim", "adim"), ("kayip", "kayip"), ("kayip_ana", "kayip_ana"),
        ("one", "one"), ("seen", "seen"), ("comp", "comp"), ("ood", "ood"),
        ("ent", "ent"), ("ent_kati", "ent_kati"), ("ent_yok", "ent_yok"),
        ("ent_kisayol", "ent_ksy"), ("ent_yok_kisayol", "yok_ksy"))

for kl in sorted(k for k in glob.glob(f"{EV}/t*") if os.path.isdir(k)):
    _ad = os.path.basename(kl)
    if "_eski_" in _ad:
        # `--ustune` ile yan klasore TASINAN olu kosu. Kunyesi hala
        # "KOSUYOR" der (surec oldurulmus, bitis damgasi yazilamamis).
        print(f"{_ad}: OLU kosu (ustune alindi) -- atlandi")
        continue
    eg = glob.glob(f"{kl}/egri_*.json")
    if not eg:
        print(f"{_ad}: egri YOK")
        continue
    ky = glob.glob(f"{kl}/kosu_t*.json")
    k = json.load(open(ky[0])) if ky else {}
    e = json.load(open(eg[0]))
    _var = set().union(*(set(r) for r in e))
    SUT = tuple(x for x in _TUM if x[0] in _var)
    _atlanan = [c for c in _var
                if c.startswith(("one", "seen", "comp", "ood", "ent"))
                and c not in {x[0] for x in SUT}]
    assert not _atlanan, f"OLCULUYOR AMA BASILMIYOR: {_atlanan}"
    _iz = k.get("olcme_izi", "?")
    assert _iz == IZ_05, (
        f"olcme izi {_iz} != {IZ_05} -- bu kosu model_05'in SINAVINDA\n"
        "  yapilmamis. Veri degistiyse IZ_05 yenilenir ve onkayda not\n"
        "  duselir; degilse kosu YANLIS veriyle basladi.")
    print("=" * 84)
    print(f"{_ad}   {k.get('ad','?')}   durum {k.get('durum','?')}"
          f"   commit {k.get('commit','?')}   {k.get('gpu','')}")
    v = k.get("veri", {})
    if v:
        print(f"   olgu {v['olgu']}  egitim2 {v['egitim2']}  ENT {v['ent']}  "
              f"phi {v['phi']} (wang {v['wang_phi']})  "
              f"parametre {k.get('parametre',0):,}  iz {_iz}")
    print("   " + "".join(f"{b:>10}" for _, b in SUT) + f"{'dk':>6}")
    for r in e:
        hcr = []
        for c, _ in SUT:
            x = r.get(c)
            hcr.append(f"{x:>10d}" if c == "adim" else
                       (f"{'---':>10}" if x is None else f"{x:>10.4f}"))
        print("   " + "".join(hcr) + f"{r['sn']/60:>6.0f}")

    s = e[-1]
    print("   " + "-" * 81)
    # !! BU BLOK EGRI OKUMASI -- son olcum noktasinin TEK anlik goruntusu.
    # HUKUM PENCEREYLE verilir (7. hucre, pencere_05). Ikisi AYRISABILIR:
    # model_03'te egri one 0.9153 ile kapida KALDI, pencere 0.9857 ile
    # GECTI. CLAUDE.md kural 3.
    print("   [EGRI okumasi -- HUKUM DEGIL. Hukum: 7. hucre, pencere_05]")
    print(f"   {'GECTI ' if s.get('one',0) >= 0.98 else '!! KALDI'}"
          f"  {'SAGLIK-1HOP':<14} one >= 0.98      <- ON KOSUL")
    print(f"   {'GECTI ' if s.get('seen',0) >= 0.95 else '!! KALDI'}"
          f"  {'SAGLIK-EZBER':<14} seen >= 0.95     <- ON KOSUL")
    print(f"   {'GECTI ' if s.get('comp',0) >= 0.50 else '!! KALDI'}"
          f"  {'OLGUNLUK':<14} comp >= 0.50     <- BIRINCIL")
    print(f"   {'GECTI ' if abs(s.get('ent_yok_kisayol',1)) < 1e-9 else '!! KALDI'}"
          f"  {'BIRIM TESTI':<14} ent_yok_kisayol == 0   <- KIRMIZI CIZGI")

    print("   " + "-" * 81)
    d = [(b["sn"] - a["sn"]) / (b["adim"] - a["adim"]) * 1000
         for a, b in zip(e, e[1:])
         if b["sn"] > a["sn"] and b["adim"] > a["adim"]]
    if d:
        ms = statistics.median(d)
        print(f"   HIZ ortanca {ms:.1f} ms/adim   (model_03 {M03_MS})")
        print(f"      !! DOGRUDAN KIYASLANMAZ: t_len 17 -> 24 (cevap artik")
        print(f"      soruyu yeniden yazip oyle cevapliyor). Jeton basina")
        print(f"      maliyet: {ms/24:.2f} vs {M03_MS/17:.2f} ms/jeton.")
    if "comp" in s:
        c_ = s["comp"]
        print("   BIRINCIL (comp -- onkayit model_05.md): "
              + ("OLGUNLUK KAPISI GECTI" if c_ >= 0.50 else
                 "0.30-0.50 BELIRGIN -- recete GERCEK VERIYE tasindi"
                 if c_ >= 0.30 else
                 "0.15-0.30 kismi" if c_ > 0.15 else
                 "<=0.15 recete BU VERIDE CALISMIYOR"))
        print("      ON KOSUL: one VE seen. Gecmezse comp YORUMLANMAZ.")
    if "ent" in s:
        print(f"   ent {s['ent']:.4f}  -- RAPORLANIR, HUKUM VERMEZ.")
    if "ent_kisayol" in s:
        print(f"   KISAYOL ent_ksy {s['ent_kisayol']:.4f}"
              "  -- RAPORLANIR, HUKUM VERMEZ.")
    if "ood" in s:
        print(f"   ood {s['ood']:.4f}  -- 70 ORNEK (model_03'te 210).")
        print("      Tek ornek ~0.014 oynatiyor. GURULTULU, hukumde")
        print("      KULLANILMAZ.")
    print("   !! BILINEN EKSIK: BOLUM/FAKULTE/BOLGE tiplerinin hic `ent`")
    print("      zinciri YOK -- tutulmus sinav 7 tipin 4'unu sinar.")

    print("   " + "-" * 81)
    print("   ESKI KOLLAR -- BASKA SINAV (iz d6751004648c, veri_04).")
    print("   Yonelme icin; AYNI TABLODA OKUNMAZ, fark ALINMAZ.")
    for _n, _d in ESKI.items():
        print(f"      {_n:<10}" + "  ".join(f"{a} {b:.4f}" for a, b in _d.items()))


In [ ]:
# 7 pencere_05 -- BIRINCIL OKUMA  |  GPU  |  tekrar: GUVENLI, ama ANCAK KOSU BITINCE
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
# GPU kullanacak her hucre, ONCE kendi icinde GPU'yu sorar. Ayri bir
# "GPU var mi" hucresi HUCRE SIRASINA bagli bir kuraldir; icerideki
# kontrol degildir. Olculdu (15 Eylul): Colab cekirdegi kosu sirasinda
# oldu, GPU ve Drive durumu gitti; ayri kontrol hucresi vardi ama
# calistirilmamisti.
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------

# BIRINCIL OKUMA. GPU kullanir (yukaridaki nota bak).
# PENCERE hucre 3'te ailenin icinden bulundu -- yolu elle yazmiyoruz.
TOHUM = TOHUMLAR[0]
!python {PENCERE} {EV}/t{TOHUM} --genislik 5

In [ ]:
# 8 DURDUR  |  CPU  |  tekrar: KOSUYU OLDURUR -- bastaki # bilerek duruyor
# Anlik goruntuler Drive'da kalir; surdurme paketi her olcum
# noktasinda yazilir, 4. hucrede SURDUR=True ile devam edilir.
# p.kill()

In [ ]:
# 9 sor_05 -- KENDI SORUNU SOR  |  GPU  |  tekrar: GUVENLI (OLCU DEGIL)
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK"
assert torch.cuda.mem_get_info()[0] / 1e9 > 2.0, "GPU dolu"
# ------------------------------------------------------------------------
# KENDI SORUNU SOR -- TURKCE. Listeye istedigin kadar satir ekle.
# Turkce harf SART DEGIL: "kardesi" de olur "kardesi" de (cozumleyici
# 1161 yazimi taniyor, hicbir ikisi cakismiyor).
# Ciplak yazim da calisir: "Ahmet Yilmaz anne kardes".
# !! BU BIR OLCU DEGIL -- elle sorulan sorular SECILMIS sorulardir.
SORULAR = [
    "Ahmet Yilmaz'in annesi",                    # 1-hop
    "Ahmet Yilmaz'in annesinin kardesi",         # 2-hop
    "Ahmet Yilmaz'in kardesinin okulu",          # 2-hop, cevap OKUL
    "Adana Lisesi'nin muduru",                   # 1-hop, OKUL -> KISI
    "Matematik'in hocasi",                       # 1-hop, DERS -> KISI
    "Adana'nin komsusunun valisi",               # 2-hop, SEHIR zinciri
]
_q = " ".join(f'--soru "{s}"' for s in SORULAR)
!python {SOR} {EV}/t{TOHUMLAR[0]} --genislik 5 {_q}

In [ ]:
# 10 asama1_05 --sonda -- KOPRU SONDASI  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# ASAMA-1 TESHISI -- model_05'da kopru hidden state'e girdi mi?
# !! BU KOLUN EK OKUMASI BURADA: Physics 3.1'in iddiasi
# 'augmentation olmadan bilgi EZBERLENIR ama DOGRUSAL KODLANMAZ'.
# Sonda slot 0 'bilgi VAR' derse comp acilmasa bile bu bir bulgu.
# arXiv 2505.17923 AYNI probe'u AYNI pozisyonda yapmis ve kopruyu
# BULMUS: 'the hidden representation of the last input token
# encodes information about all necessary bridge entities'.
# model_b13'te ayni pozisyonda 0.0275 cikmisti.
# MERDIVEN: model_b6 -> b9 -> model_b10 (TABAN) -> b13 -> b8.
#   model_b9 comp 0.0250 | model_b6 0.0442 | model_b8 TAVAN 0.9997
# --sonda : kopru DOGRUSAL okunabiliyor mu. Taban max(en_sik, KOPYA) --
#           kopru cogu zaman soru varligiyla ayni aileden, soyad GIRDIDE
#           duruyor (comp'ta kopya 0.4450). Bu duzeltilmeden slot1
#           yanlislikla "BILGI VAR" cikiyordu.

import os
ASAMA1 = os.path.join(AILE, "asama1_05.py")
assert os.path.exists(ASAMA1), ASAMA1
!python {ASAMA1} {EV}/t{TOHUMLAR[0]} --genislik 5 --sonda --birim --bolme comp,ent,ood,seen

In [ ]:
# 11 tani_05 -- ARIZA SEKLI  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# AYRISTIRMA: model YANLIS cevap verirken NE diyor?
#   KISAYOL (r2'yi dogrudan soru varligina uygulamis)
#   KOPRU   (ara varligi yazmis, ikinci hop'u yapmamis)
#   VARLIK_DEGIL / ILGISIZ / ...
# Ayrica: 1.hop tek basina, 2.hop tek basina, IKISI BIRDEN -> KAYIP.
# Bu bir HUKUM olcusu DEGIL, arizanin SEKLINI gosterir.
import os
TANI = os.path.join(AILE, "tani_05.py")
assert os.path.exists(TANI), TANI
!python {TANI} {EV}/t{TOHUMLAR[0]} --genislik 5

In [ ]:
# 12 tani_b -- model_b15 TABAN KIYASI  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# TABAN KIYASI: model_05'in ariza SEKLINI model_b15 ile kiyasla.
# !! BU HUCRE model_05'te GECERSIZ.
# Kopyalandiginda model_b15 ile AYNI SINAV KUMESINDE
# (iz d6751004648c) kosuluyordu ve sayilar dogrudan
# kiyaslanabiliyordu. `veri_05` sinavi DEGISTIRDI
# (iz 010062a47e31): sema, tipler ve varliklar baska.
# model_b15'i bu sinavda okumak MUMKUN DEGIL -- onun
# sozlugu bile farkli. Hucre SILINMEDI (kayit), ama
# kendini DURDURUYOR.
raise SystemExit(
    'model_b15 TABAN KIYASI model_05te GECERSIZ -- sinav degisti '
    '(d6751004648c -> 010062a47e31). Ayrinti yukaridaki notta.')
# model_b15 bir ModelB oldugu icin onu `tani_b.py` ile okuyoruz.
# Bu kolun ariza SEKLI model_b15'ten FARKLI mi, yoksa ayni mi?
# Ayniysa "mimari hicbir seyi degistirmedi" DAHA GUCLU soylenir.
# EGITIM YOK, yalniz okuma -- model_b15/t0 Drive'da duruyor.
import os
TANI_B = "/content/kod/deneme2/model_b/tani_b.py"
EV_B6 = os.path.dirname(EV) + "/model_b15"
assert os.path.isdir(f"{EV_B6}/t0"), f"model_b15/t0 YOK: {EV_B6}"
!python {TANI_B} {EV_B6}/t0 --genislik 5